# 🚗  25COA207 CW1 Portfolio - Route Finding

This is the notebook containing the code implementation of the route finding AI search system, the logic for the system is contained within the logic section below. To use the AI system, use the ouput block at the bottom.

# 🧠 Logic
The logic which powers the AI system.

## Set Up
This sections is where the ipywidgets library used for GUI is imported, and where the map (the environment for the AI system) is defined in the form of an adjacency list.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import random

# This section defines an adjacency list for the premade map, this is the version without any traffic
mapTemplate = {
    "Holywell": {"West Entrance": 200},
    "Holywell Gym": {"STEM Building": 150},
    "West Entrance": {"Holywell": 200, "STEM Building": 125},
    "STEM Building": {"Holywell Gym": 150, "West Entrance": 125, "Engineering Building": 130},
    "Car Park": {"Engineering Building": 100, "Robert Bakewell": 300},
    "Engineering Building": {"STEM Building": 130, "Car Park": 100, "David Collett": 200},
    "David Collett": {"Engineering Building": 200, "Pilkington Library": 140, "Robert Bakewell": 200},
    "Pilkington Library": {"David Collett": 140, "Robert Bakewell": 125, "South Entrance": 380},
    "Robert Bakewell": {"Car Park": 300, "David Collett": 200, "Pilkington Library": 125},
    "Elvyn Richards": {"Robert Bakewell": 75},
    "Telford": {"Pilkington Library": 125},
    "Hazlerigg-Rutland": {"Elvyn Richards": 75, "Cayley": 225},
    "Faraday": {"Telford": 175},
    "Co-Op": {"Faraday": 65, "Cayley": 90, "Lacrosse Pitch": 150},
    "Cayley": {"Hazlerigg-Rutland": 225, "Co-Op": 90, "Claudia Parsons": 200},
    "Claudia Parsons": {"Cayley": 200},
    "South Entrance": {"Pilkington Library": 380, "Lacrosse Pitch": 160},
    "Edward Herbert Building": {"Lacrosse Pitch": 215},
    "Haslegrave Building": {"Lacrosse Pitch": 215, "James France Building": 250, "Fountain": 185},
    "James France Building": {"Haslegrave Building": 250, "Towers": 245},
    "Fountain": {"Haslegrave Building": 185, "Student Union": 100},
    "Student Union": {"Fountain": 100, "East Entrance": 140},
    "Lacrosse Pitch": {"Co-Op": 150, "South Entrance": 160, "Edward Herbert Building": 215, "Haslegrave Building": 215},
    "East Entrance": {"Fountain": 200, "Student Union": 140},
    "Towers": {"James France Building": 245, "Martin Hall": 170},
    "Martin Hall": {"Fountain": 375, "Towers": 170}
}

currentMap = {}

# CopyTo is a helper function that copies a version of a map, instead of causing issues throguh passing paramaters by reference
def copyTo(clipboard):
  target = {}

  for node in clipboard:
    target[node] = clipboard[node].copy()

  return target

map = copyTo(mapTemplate)

## Traffic
This section will implement the traffic compllication.


In [ ]:
# This procedure adds traffic to the predefined map and copies in to the current map
def addTraffic():
  global currentMap

  mapTraffic = {}

  mapTraffic = copyTo(mapTemplate)

  # The random seed used for traffic is hard coded for repeatability and testing
  random.seed("25COA207")

  for node in mapTemplate:
    for neighbour in mapTemplate[node]:
      # Random traffic amount created to be added to the edge
      modifier = random.randint(0,100)

      mapTraffic[node][neighbour] = mapTemplate[node][neighbour] + modifier

      # Makes sure that the modifier is added to both directions of an edge
      if node in mapTemplate[neighbour]:
        mapTraffic[neighbour][node] = mapTemplate[neighbour][node] + modifier

  currentMap = copyTo(mapTraffic)


## Route Finding Algorithm
This is the algorithm that solves the route between two points, it is enclosed within a procedure to allow for reuse.

In [ ]:
# This is the algorithm which handles sovling the fastest route between two nodes in the map
def solveRoute(b):
  global currentMap

  # Determine whether traffic is enabled or not
  if traffic.value:
    addTraffic()
  else:
    currentMap = copyTo(mapTemplate)

  if outputMaps.value:
    print(f"\nCurrent Map: {currentMap}")

  startNode = startDropdown.value
  endNode = endDropdown.value

  visited = []
  costs = {}

  # Pre-set all the costs other than the start node to a very high number
  for node in currentMap:
    costs[node] = (9999, "")

  costs[startNode] = (0, "")

  currentNode = startNode
  previousNode = None

  # Iterate until the every node has been visited
  while len(visited) < len(currentMap):
    for neighbour in currentMap[currentNode]:
      neighbourCost = costs[currentNode][0] +currentMap[currentNode][neighbour]
      if neighbourCost < costs[neighbour][0]:
        # If newly calculated cost to reach neighbour is less than current one update it, also store the current node as the previous node
        costs[neighbour] = [neighbourCost, currentNode]
    visited.append(currentNode)

    lowest = 9999

    # Select the lowest unvisited node as the new current node
    for cost in costs:
      if costs[cost][0] < lowest and cost not in visited:
        lowest = costs[cost][0]
        currentNode = cost

  complete = False

  # Reassemble the route from the costs list, using the previous node stored next to cost as a linked list
  route = []

  # Start at the end node set earlier
  selectedNode = endNode

  while selectedNode != startNode:
    # Nodes are added to the route in reverse order
    route = [selectedNode] + route
    selectedNode = costs[selectedNode][1]
  route = [startNode] + route

  print("\nRoute:")
  print(*route, sep=" ➡ ")
  print(f"Route Cost: {costs[endNode][0]}m, Route Time: {round((costs[endNode][0]/4.44)//60)}m {round((costs[endNode][0]/4.44)%60)}s")

## GUI Set Up
This section sets up the code for the basic GUI.

In [ ]:
# Create dropdown for start node
startDropdown = widgets.Dropdown(
    options=sorted(map.keys()),
    description="Start:"
)

# Create dropdown for end node
endDropdown = widgets.Dropdown(
    options=sorted(map.keys()),
    description="End:"
)

button = widgets.Button(description="Find Route")

# Create toggle for traffic
traffic = widgets.Checkbox(description="Enable traffic", value=False)

# A toggle to output the current map, used for debugging and testing
outputMaps = widgets.Checkbox(description="Output Map", value=False)

button.on_click(solveRoute)

# 📺 Output
The section which outputs the interface for the user.

In [ ]:
display(startDropdown)
display(endDropdown)
display(traffic)
display(outputMaps)
display(button)